In [ ]:
import time
import gc
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from hep_tracking.dataset import TrackDataset
from hep_tracking.config import KNNModelConfig, ClassifierModelConfig
from hep_tracking.models import (
    ScipyCKDTree, 
    SklearnKNN, 
    FaissExact, 
    FaissIVFFlat, 
    FaissIVFPQ, 
    HnswGraph
)
from hep_tracking.pipeline import PipelineEvaluator
from hep_tracking.features import compute_pair_features, create_pair_dataset, split_by_event
from hep_tracking.classifiers import XGBoostWrapper, LightGBMWrapper, RandomForestWrapper
from hep_tracking.plots import (
    plot_silver_bullet, 
    plot_time_vs_size, 
    plot_time_breakdown, 
    plot_purity_vs_efficiency
)

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.figsize': (10, 6)})

In [ ]:
# --- Zbiory danych do ewaluacji ---
DATASETS_TO_TEST = [
    "hard_10k", 
    "hard_100k", 
    "hard_1M"
]

# --- Konfiguracja wyszukiwarek kandydatów---
RETRIEVERS_CONFIG = [
    # --- Dokładne (Exact kNN) ---
    KNNModelConfig("cKDTree (Exact CPU)", ScipyCKDTree, {"workers": -1}),
    # KNNModelConfig("Sklearn KDTree (Exact CPU)", SklearnKNN, {"algorithm": "kd_tree", "leaf_size": 100}),
    # KNNModelConfig("Sklearn BallTree (Exact CPU)", SklearnKNN, {"algorithm": "ball_tree", "leaf_size": 100}),
    # KNNModelConfig("Faiss FlatL2 (Exact GPU)", FaissExact, {"use_gpu": True}),
    
    # --- Przybliżone (ANN) ---
    KNNModelConfig("HNSW (ANN CPU)", HnswGraph, {"m": 16, "ef_construction": 200, "ef": 50}),
    KNNModelConfig("IVFFlat (ANN GPU)", FaissIVFFlat, {"nprobe": 20, "use_gpu": True}),
    # KNNModelConfig("IVFPQ (ANN GPU)", FaissIVFPQ, {"nprobe": 20, "m": 5, "nbits": 8, "use_gpu": True}),
]

# --- Konfiguracja Trenowania Klasyfikatora ML (XGBoost) ---
TRAIN_CLASSIFIER = True
TRAIN_DATASET_NAME = "hard_100k"

# Wybierz klasyfikator, odkomentowując jedną z poniższych konfiguracji:

CLASSIFIERS_CONFIG = [
    ClassifierModelConfig("XGBoost", XGBoostWrapper, {"n_estimators": 100, "learning_rate": 0.1, "max_depth": 6, "n_jobs": -1}),
    # ClassifierModelConfig("LightGBM", LightGBMWrapper, {"n_estimators": 100, "learning_rate": 0.1, "max_depth": 6, "n_jobs": -1}),
    # ClassifierModelConfig("Random Forest", RandomForestWrapper, {"n_estimators": 100, "max_depth": 10, "n_jobs": -1})
]

# --- Parametry potoku---
K_NEIGHBORS = 5
ML_PROB_THRESHOLD = 0.5

# Cięcia geometryczne dla potoku bez ML
GEOM_CUTS = {
    "max_delta_r": 20.0,    # Maksymalna różnica promienia
    "min_dot_product": 0.95 # Minimalny iloczyn skalarny
}

# --- Wymiary do przetestowania ---
DIMENSIONS_TO_TEST = [2, 4, 5, 8]

# --- Flagi uruchomieniowe  ---
RUN_GEOMETRIC_PIPELINE = True
RUN_ML_PIPELINE = False
RUN_ALL_PAIRS_BASELINE = False

In [ ]:
if TRAIN_CLASSIFIER:
    print(f"=== Rozpoczęcie przygotowania modelu na zbiorze: {TRAIN_DATASET_NAME} ===")
    train_path = DATA_DIR / f"dataset_{TRAIN_DATASET_NAME}.npz"
    train_dataset = TrackDataset.from_npz(train_path)

    print("1. Wyszukiwanie sąsiadów do budowy par uczących (cKDTree)...")
    ref_knn = ScipyCKDTree(workers=-1)
    ref_knn.fit(train_dataset.X)
    _, train_candidates = ref_knn.kneighbors(train_dataset.X, k=K_NEIGHBORS)

    print("2. Ekstrakcja cech fizycznych dla par...")
    t0_feat = time.perf_counter()
    dataset_pairs = create_pair_dataset(train_dataset, train_candidates, max_neg_ratio=5)
    time_feat = time.perf_counter() - t0_feat
    print(f"   -> Zbudowano zbiór {len(dataset_pairs.y)} par w {time_feat:.2f} s")

    train_pairs, val_pairs, _ = split_by_event(
        dataset=dataset_pairs, 
        train_size=0.8, 
        val_size=0.2, 
        seed=42
    )
    print(f"   -> Rozkład par - Train: {len(train_pairs.y)}, Val: {len(val_pairs.y)}")

    print("3. Trening klasyfikatorów ML...")
    trained_classifiers = {}
    
    for clf_cfg in CLASSIFIERS_CONFIG:
        print(f"   -> Trenowanie: {clf_cfg.name}...")
        clf = clf_cfg.model_factory(**clf_cfg.model_kwargs)
        
        t0_train = time.perf_counter()
        clf.fit(train_pairs.X, train_pairs.y, X_val=val_pairs.X, y_val=val_pairs.y)
        time_train = time.perf_counter() - t0_train
        
        trained_classifiers[clf_cfg.name] = clf
        print(f"      Zakończono w {time_train:.2f} s")
        
    print("=== ZAKOŃCZONO ETAP PRZYGOTOWAŃ ML ===")
else:
    trained_classifiers = {}
    print("Pomijam przygotowanie ML (Flaga TRAIN_CLASSIFIERS = False).")

In [ ]:
from IPython.display import display

print("\n=== ROZPOCZĘCIE TESTÓW POTOKÓW END-TO-END ===")
all_results = []

for dataset_name in DATASETS_TO_TEST:
    print(f"\nPrzetwarzanie zbioru: {dataset_name}")
    ds_path = DATA_DIR / f"dataset_{dataset_name}.npz"
    if not ds_path.exists():
        print(f"  [UWAGA] Plik {ds_path.name} nie istnieje. Pomijam.")
        continue
        
    full_dataset = TrackDataset.from_npz(ds_path)
    unique_events = np.unique(full_dataset.event_ids)
    print(f"  Znaleziono {len(unique_events)} zdarzeń (events) w zbiorze.")
    
    for event_id in unique_events:
        mask = full_dataset.event_ids == event_id
        event_ds = TrackDataset(
            X=full_dataset.X[mask],
            y=full_dataset.y[mask],
            event_ids=full_dataset.event_ids[mask]
        )
        
        n_hits = len(event_ds.X)
        nlist = min(100, int(np.sqrt(n_hits)))
        
        evaluator = PipelineEvaluator(
            dataset=event_ds, k_neighbors=K_NEIGHBORS, 
            geom_cuts=GEOM_CUTS, ml_threshold=ML_PROB_THRESHOLD
        )
        
        for target_dim in DIMENSIONS_TO_TEST:
            if target_dim < 5:
                pca = PCA(n_components=target_dim, random_state=42)
                X_retrieval = pca.fit_transform(event_ds.X).astype(np.float32)
            elif target_dim > 5:
                padding = np.zeros((n_hits, target_dim - 5), dtype=np.float32)
                X_retrieval = np.hstack([event_ds.X, padding])
            else:
                X_retrieval = event_ds.X
                
            for retriever_cfg in RETRIEVERS_CONFIG:
                kwargs = retriever_cfg.model_kwargs.copy()
                
                if "IVF" in retriever_cfg.name:
                    retriever_cfg.model_kwargs["nlist"] = nlist

                if "IVFPQ" in retriever_cfg.name and "m" in kwargs:
                    if target_dim % kwargs["m"] != 0:
                        for cand_m in range(kwargs["m"], 0, -1):
                            if target_dim % cand_m == 0:
                                kwargs["m"] = cand_m
                                break
                
                retriever = retriever_cfg.model_factory(**retriever_cfg.model_kwargs)
                pipeline_prefix = f"[{target_dim}D] {retriever_cfg.name}"
                
                if RUN_GEOMETRIC_PIPELINE:
                    res = evaluator.run_geometric_pipeline(retriever, X_retrieval=X_retrieval)
                    res.update({
                        "Dataset": dataset_name, "Event_ID": event_id, "Hits": n_hits, "Dim": target_dim,
                        "Retriever": retriever_cfg.name, "Filter": "Geometric Cuts",
                        "Pipeline": f"{pipeline_prefix} + GeomCuts"
                    })
                    all_results.append(res)
                    
                if RUN_ML_PIPELINE and trained_classifiers:
                    for clf_name, clf_model in trained_classifiers.items():
                        res = evaluator.run_ml_pipeline(retriever, clf_model, X_retrieval=X_retrieval)
                        res.update({
                            "Dataset": dataset_name, "Event_ID": event_id, "Hits": n_hits, "Dim": target_dim,
                            "Retriever": retriever_cfg.name, "Filter": clf_name,
                            "Pipeline": f"{pipeline_prefix} + {clf_name}"
                        })
                        all_results.append(res)
        
        if RUN_ALL_PAIRS_BASELINE and trained_classifiers:
            if n_hits > 15000:
                print(f"  [POMINIĘTO] Zdarzenie {event_id} dla All-Pairs (zbyt wiele hitów: {n_hits}).")
            else:
                for clf_name, clf_model in trained_classifiers.items():
                    res = evaluator.run_all_pairs_pipeline(clf_model)
                    res.update({
                        "Dataset": dataset_name, "Event_ID": event_id, "Hits": n_hits, "Dim": 5, # Zawsze na oryginalnych
                        "Retriever": "None (All Pairs)", "Filter": clf_name,
                        "Pipeline": f"All-Pairs + {clf_name}"
                    })
                    all_results.append(res)

df_results = pd.DataFrame(all_results)
if not df_results.empty:
    df_results["Physics_Quality"] = df_results["Purity"] * df_results["Efficiency"]

print("\n=== ZAKOŃCZONO WSZYSTKIE TESTY ===")
display(df_results.head())

In [ ]:
summary_df = df_results.groupby(["Dataset", "Pipeline"]).agg(
    Purity_Mean=("Purity", "mean"),
    Efficiency_Mean=("Efficiency", "mean"),
    Physics_Quality_Mean=("Physics_Quality", "mean"),
    Time_Retrieval_Mean=("Time_Retrieval_s", "mean"),
    Time_Features_Mean=("Time_Features_s", "mean"),
    Time_Filter_Mean=("Time_Filter_s", "mean"),
    Time_Total_Mean=("Time_Total_s", "mean")
).reset_index().sort_values(by="Physics_Quality_Mean", ascending=False)

print("\n=== TABELA PODSUMOWUJĄCA: ŚREDNIE CZASY I METRYKI NA ZDARZENIE ===")
display(summary_df.style.format({
    "Purity_Mean": "{:.3f}", "Efficiency_Mean": "{:.3f}", "Physics_Quality_Mean": "{:.3f}",
    "Time_Retrieval_Mean": "{:.4f} s", "Time_Features_Mean": "{:.4f} s",
    "Time_Filter_Mean": "{:.4f} s", "Time_Total_Mean": "{:.4f} s"
}).background_gradient(subset=["Physics_Quality_Mean"], cmap="RdYlGn"))

print("\n=== GENEROWANIE WYKRESÓW ===")
plots = {
    "silver_bullet": plot_silver_bullet,
    "time_vs_size": plot_time_vs_size,
    "time_breakdown": plot_time_breakdown,
    "purity_vs_efficiency": plot_purity_vs_efficiency
}

for name, plot_func in plots.items():
    out_path = PROJECT_ROOT / f"{name}_plot.pdf"
    plot_func(df_results, output_path=str(out_path))
    print(f" Zapisano: {out_path.name}")

print("\nGotowe! Wszystkie materiały do raportu zostały wygenerowane.")